In [1]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 11.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.4 MB/s eta 0:00:00:00:0100:01


In [2]:
import os, gc, json, ast, random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset, load_dataset

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

SEED         = 42
EVAL_SAMPLES = 5000
N_BINS       = 10
EPOCHS       = 3

MODEL_PATH   = '/kaggle/input/models/metaresearch/llama-2/pytorch/7b-hf/1'
DATA_PATH    = '/kaggle/working/datasets'
RESULTS_PATH = '/kaggle/working/results'
CKPT_PATH    = '/kaggle/working/checkpoints'

os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(CKPT_PATH,    exist_ok=True)
os.makedirs(f'{DATA_PATH}/dolly', exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
print("All imports done.")

CUDA available: True
GPU count: 2
  GPU 0: Tesla T4 | 15.6 GB
  GPU 1: Tesla T4 | 15.6 GB
All imports done.


In [3]:
# ============================================================
# NOTEBOOK 1: DATASET PREPARATION
# Run this once — saves all datasets to /kaggle/working/datasets/
# ============================================================

import os
import json
import random
import pandas as pd
import numpy as np
import requests
from datasets import load_dataset

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Directory setup
base_path = '/kaggle/working/datasets'
os.makedirs(base_path, exist_ok=True)
os.makedirs(f'{base_path}/alpaca', exist_ok=True)
os.makedirs(f'{base_path}/oa', exist_ok=True)

print("="*60)
print("DATASET PREPARATION")
print("="*60)

# ============================================================
# 1. PILE DATASET (Streaming, 50k samples)
# ============================================================
print("\n[1/5] Loading PILE dataset (streaming)...")

pile_save_path = f'{base_path}/pile_50k.csv'

if os.path.exists(pile_save_path):
    print(f"   ✅ Already exists: {pile_save_path}")
    pile_df = pd.read_csv(pile_save_path)
    print(f"   Samples: {len(pile_df)}")
else:
    parts = [
        {'skip': 0,      'target': 17000, 'label': 'Start'},
        {'skip': 200000, 'target': 17000, 'label': 'Middle'},
        {'skip': 500000, 'target': 16000, 'label': 'End'},
    ]

    all_samples = []

    for part in parts:
        print(f"\n   📥 Collecting from {part['label']} (skip={part['skip']})...")

        dataset = load_dataset(
            "monology/pile-uncopyrighted",
            split="train",
            streaming=True
        )

        samples = []
        skipped = 0
        processed = 0

        for example in dataset:
            if skipped < part['skip']:
                skipped += 1
                continue

            text = example.get('text', '').strip()
            if len(text) > 100:
                samples.append({
                    'text': text,
                    'source': example.get('meta', {}).get('pile_set_name', 'unknown')
                })
                processed += 1

            if processed >= part['target']:
                break

            if processed % 5000 == 0 and processed > 0:
                print(f"      Collected {processed}/{part['target']}...")

        print(f"   ✅ Collected {len(samples)} from {part['label']}")
        all_samples.extend(samples)

    print(f"\n   🔀 Shuffling {len(all_samples)} samples...")
    random.shuffle(all_samples)

    pile_df = pd.DataFrame(all_samples[:50000])
    pile_df.to_csv(pile_save_path, index=False)
    print(f"   ✅ PILE saved: {len(pile_df)} samples → {pile_save_path}")
    print(f"   Source distribution:\n{pile_df['source'].value_counts().head(5)}")

# ============================================================
# 2. T-REx DATASET (KILT, streaming download)
# ============================================================
print("\n[2/5] Loading T-REx (KILT) dataset...")

trex_save_path = f'{base_path}/trex_50k.csv'

if os.path.exists(trex_save_path):
    print(f"   ✅ Already exists: {trex_save_path}")
    trex_df = pd.read_csv(trex_save_path)
    print(f"   Samples: {len(trex_df)}")
else:
    url = "http://dl.fbaipublicfiles.com/KILT/trex-train-kilt.jsonl"
    print(f"   Downloading from: {url}")
    response = requests.get(url, stream=True)

    samples = []
    target = 50000
    raw_lines = 0

    for line in response.iter_lines():
        if len(samples) >= target:
            break
        if not line:
            continue

        raw_lines += 1
        try:
            item = json.loads(line)
            input_text = item.get('input', '').strip()
            outputs = item.get('output', [])
            if not outputs:
                continue
            answer = outputs[0].get('answer', '').strip()
            if not answer:
                continue
            if '[SEP]' not in input_text:
                continue

            parts = input_text.split('[SEP]')
            subject = parts[0].strip()
            relation = parts[1].strip() if len(parts) > 1 else ''
            text = f"{subject}'s {relation} is"
            entity = answer

            samples.append({
                'text': text,
                'entity': entity,
                'subject': subject,
                'relation': relation
            })

        except Exception:
            continue

        if raw_lines % 100000 == 0:
            print(f"   Processed {raw_lines} lines, collected {len(samples)}...")

    response.close()
    trex_df = pd.DataFrame(samples[:target])
    trex_df.to_csv(trex_save_path, index=False)
    print(f"   ✅ T-REx saved: {len(trex_df)} samples → {trex_save_path}")

# ============================================================
# 3. MMLU DATASET
# ============================================================
print("\n[3/5] Loading MMLU dataset...")

mmlu_test_path = f'{base_path}/mmlu_test.csv'
mmlu_val_path  = f'{base_path}/mmlu_val.csv'

if os.path.exists(mmlu_test_path) and os.path.exists(mmlu_val_path):
    print(f"   ✅ Already exists")
    mmlu_df     = pd.read_csv(mmlu_test_path)
    mmlu_val_df = pd.read_csv(mmlu_val_path)
    print(f"   Test samples:  {len(mmlu_df)}")
    print(f"   Val samples:   {len(mmlu_val_df)}")
else:
    mmlu     = load_dataset("cais/mmlu", "all", split="test")
    mmlu_val = load_dataset("cais/mmlu", "all", split="validation")

    mmlu_df     = pd.DataFrame(mmlu)
    mmlu_val_df = pd.DataFrame(mmlu_val)

    mmlu_df.to_csv(mmlu_test_path, index=False)
    mmlu_val_df.to_csv(mmlu_val_path, index=False)
    print(f"   ✅ MMLU test saved:  {len(mmlu_df)} samples")
    print(f"   ✅ MMLU val saved:   {len(mmlu_val_df)} samples")
    print(f"   Subjects: {mmlu_df['subject'].nunique()}")

# ============================================================
# 4. ALPACA DATASET (Fine-tuning data)
# ============================================================
print("\n[4/5] Loading Alpaca dataset...")

alpaca_path = f'{base_path}/alpaca/alpaca_data.csv'

if os.path.exists(alpaca_path):
    print(f"   ✅ Already exists")
    alpaca_df = pd.read_csv(alpaca_path)
    print(f"   Samples: {len(alpaca_df)}")
else:
    alpaca_raw = load_dataset("tatsu-lab/alpaca", split="train")

    def format_alpaca(example):
        if example["input"] and example["input"].strip():
            text = (
                f"### Instruction:\n{example['instruction']}\n"
                f"### Input:\n{example['input']}\n"
                f"### Response:\n{example['output']}"
            )
        else:
            text = (
                f"### Instruction:\n{example['instruction']}\n"
                f"### Response:\n{example['output']}"
            )
        return {"text": text}

    alpaca_formatted = alpaca_raw.map(format_alpaca)
    alpaca_df = pd.DataFrame({
        'text': alpaca_formatted['text'],
        'instruction': alpaca_formatted['instruction'],
        'input': alpaca_formatted['input'],
        'output': alpaca_formatted['output'],
    })
    alpaca_df.to_csv(alpaca_path, index=False)
    print(f"   ✅ Alpaca saved: {len(alpaca_df)} samples → {alpaca_path}")

# ============================================================
# 5. OPENASSISTANT (OA) DATASET (Fine-tuning data)
# ============================================================
print("\n[5/5] Loading OpenAssistant (OA) dataset...")

oa_path = f'{base_path}/oa/oa_data.csv'

if os.path.exists(oa_path):
    print(f"   ✅ Already exists")
    oa_df = pd.read_csv(oa_path)
    print(f"   Samples: {len(oa_df)}")
else:
    oa_raw = load_dataset("OpenAssistant/oasst1", split="train")

    # Paper: extract single-turn English conversations
    oa_samples = []
    for example in oa_raw:
        if (example.get('lang') == 'en' and
            example.get('role') == 'prompter' and
            example.get('text')):

            text = (
                f"### Instruction:\n{example['text']}\n"
                f"### Response:\n"
            )
            oa_samples.append({
                'text': text,
                'raw_text': example['text'],
                'message_id': example.get('message_id', ''),
            })

    oa_df = pd.DataFrame(oa_samples)
    # Paper uses 11k pairs — sample down
    oa_df = oa_df.sample(n=min(11000, len(oa_df)), random_state=SEED).reset_index(drop=True)
    oa_df.to_csv(oa_path, index=False)
    print(f"   ✅ OA saved: {len(oa_df)} samples → {oa_path}")

# For fair comparison, sample Alpaca to same size as OA (paper does this)
alpaca_sampled_path = f'{base_path}/alpaca/alpaca_sampled.csv'
if not os.path.exists(alpaca_sampled_path):
    alpaca_sampled = alpaca_df.sample(
        n=min(11000, len(alpaca_df)),
        random_state=SEED
    ).reset_index(drop=True)
    alpaca_sampled.to_csv(alpaca_sampled_path, index=False)
    print(f"   ✅ Alpaca sampled to {len(alpaca_sampled)} (matching OA size)")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "="*60)
print("ALL DATASETS READY")
print("="*60)
print(f"  PILE       : {pile_save_path}")
print(f"  T-REx      : {trex_save_path}")
print(f"  MMLU test  : {mmlu_test_path}")
print(f"  MMLU val   : {mmlu_val_path}")
print(f"  Alpaca     : {alpaca_path}")
print(f"  Alpaca(11k): {alpaca_sampled_path}")
print(f"  OA         : {oa_path}")

DATASET PREPARATION

[1/5] Loading PILE dataset (streaming)...

   📥 Collecting from Start (skip=0)...


README.md:   0%|          | 0.00/776 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

      Collected 5000/17000...
      Collected 10000/17000...
      Collected 15000/17000...
   ✅ Collected 17000 from Start

   📥 Collecting from Middle (skip=200000)...


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

...

data/train-00000-of-00001-b42a775f407cee(…):   0%|          | 0.00/39.5M [00:00<?, ?B/s]

data/validation-00000-of-00001-134b8fd0c(…):   0%|          | 0.00/2.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84437 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4401 [00:00<?, ? examples/s]

   ✅ OA saved: 11000 samples → /kaggle/working/datasets/oa/oa_data.csv
   ✅ Alpaca sampled to 11000 (matching OA size)

ALL DATASETS READY
  PILE       : /kaggle/working/datasets/pile_50k.csv
  T-REx      : /kaggle/working/datasets/trex_50k.csv
  MMLU test  : /kaggle/working/datasets/mmlu_test.csv
  MMLU val   : /kaggle/working/datasets/mmlu_val.csv
  Alpaca     : /kaggle/working/datasets/alpaca/alpaca_data.csv
  Alpaca(11k): /kaggle/working/datasets/alpaca/alpaca_sampled.csv
  OA         : /kaggle/working/datasets/oa/oa_data.csv


In [4]:
dolly_path = f'{DATA_PATH}/dolly/dolly_data.csv'

if os.path.exists(dolly_path):
    print("✅ Dolly already exists")
    dolly_df = pd.read_csv(dolly_path)
    print(f"   Samples: {len(dolly_df)}")
else:
    dolly_raw = load_dataset("databricks/databricks-dolly-15k", split="train")
    print(f"Raw Dolly samples: {len(dolly_raw)}")

    def format_dolly(example):
        # Dolly has: instruction, context (optional), response, category
        if example.get("context") and example["context"].strip():
            text = (
                f"### Instruction:\n{example['instruction']}\n"
                f"### Input:\n{example['context']}\n"
                f"### Response:\n{example['response']}"
            )
        else:
            text = (
                f"### Instruction:\n{example['instruction']}\n"
                f"### Response:\n{example['response']}"
            )
        return {"text": text}

    dolly_formatted = dolly_raw.map(format_dolly)
    dolly_df = pd.DataFrame({
        'text':        dolly_formatted['text'],
        'instruction': dolly_formatted['instruction'],
        'context':     dolly_formatted['context'],
        'response':    dolly_formatted['response'],
        'category':    dolly_formatted['category'],
    })

    dolly_df.to_csv(dolly_path, index=False)
    print(f"✅ Dolly saved: {len(dolly_df)} samples → {dolly_path}")
    print(f"   Category distribution:\n{dolly_df['category'].value_counts()}")

print(f"\nSample:\n{dolly_df['text'].iloc[0][:300]}")

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Raw Dolly samples: 15011


Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

✅ Dolly saved: 15011 samples → /kaggle/working/datasets/dolly/dolly_data.csv
   Category distribution:
category
open_qa                   3742
general_qa                2191
classification            2136
closed_qa                 1773
brainstorming             1766
information_extraction    1506
summarization             1188
creative_writing           709
Name: count, dtype: int64

Sample:
### Instruction:
When did Virgin Australia start operating?
### Input:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, wit


In [5]:
def parse_choices(x):
    if isinstance(x, list): return x
    try: return ast.literal_eval(x)
    except: return x

pile_df     = pd.read_csv(f'{DATA_PATH}/pile_50k.csv')
trex_df     = pd.read_csv(f'{DATA_PATH}/trex_50k.csv')
mmlu_df     = pd.read_csv(f'{DATA_PATH}/mmlu_test.csv')
mmlu_val_df = pd.read_csv(f'{DATA_PATH}/mmlu_val.csv')

mmlu_df     = mmlu_df.sample(n=3000, random_state=SEED).reset_index(drop=True)
mmlu_df['choices']     = mmlu_df['choices'].apply(parse_choices)
mmlu_val_df['choices'] = mmlu_val_df['choices'].apply(parse_choices)

pile_eval = pile_df.sample(n=EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)
trex_eval = trex_df.sample(n=EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)

ANSWER_OPTIONS = ['A', 'B', 'C', 'D']
PILE_PROMPT    = "Finish this sentence: "
TREX_PROMPT    = "Finish this Wikipedia description: "

print(f"PILE eval  : {len(pile_eval)}")
print(f"T-REx eval : {len(trex_eval)}")
print(f"MMLU test  : {len(mmlu_df)}")
print(f"MMLU val   : {len(mmlu_val_df)}")
print("Datasets ready.")

PILE eval  : 5000
T-REx eval : 5000
MMLU test  : 3000
MMLU val   : 1531
Datasets ready.


In [6]:
def compute_ece(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        if mask.sum() == 0:
            continue
        ece += (mask.sum() / len(confidences)) * abs(
            accuracies[mask].mean() - confidences[mask].mean()
        )
    return float(ece)

def compute_reliability_diagram_data(confidences, accuracies, n_bins=10):
    confidences = np.array(confidences)
    accuracies  = np.array(accuracies)
    bins = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_sizes = [], [], []
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i+1])
        bin_sizes.append(int(mask.sum()))
        if mask.sum() == 0:
            bin_accs.append(0.0)
            bin_confs.append(float((bins[i] + bins[i+1]) / 2))
        else:
            bin_accs.append(float(accuracies[mask].mean()))
            bin_confs.append(float(confidences[mask].mean()))
    return bin_accs, bin_confs, bin_sizes

print("ECE utilities ready.")

ECE utilities ready.


In [7]:
def evaluate_model(model, tokenizer, label="", include_prompt=True):
    model.eval()
    results = {}

    pile_prompt = PILE_PROMPT if include_prompt else ""
    trex_prompt = TREX_PROMPT if include_prompt else ""

    # ── TASK 1: CLM (PILE) ───────────────────────────────────
    print(f"\n  [{label}] CLM (PILE)...")
    confs, accs = [], []

    for idx, row in pile_eval.iterrows():
        text = pile_prompt + str(row['text']).strip()
        try:
            input_ids = tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=512
            )["input_ids"]
            seq_len = input_ids.shape[1]
            if seq_len < 3: continue

            pos        = np.random.randint(1, seq_len)
            context    = input_ids[:, :pos].to(model.device)
            true_token = input_ids[:, pos].item()

            with torch.no_grad():
                probs = F.softmax(
                    model(input_ids=context).logits[:, -1, :].float(), dim=-1
                )
            confs.append(probs.max(dim=-1).values.item())
            accs.append(int(probs.argmax(dim=-1).item() == true_token))
        except: continue

        if (idx+1) % 1000 == 0:
            print(f"    {idx+1}/{EVAL_SAMPLES} | "
                  f"ECE: {compute_ece(confs, accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f}")

    results['CLM'] = {
        'ECE':       compute_ece(confs, accs),
        'Accuracy':  float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ CLM ECE: {results['CLM']['ECE']:.4f} | "
          f"Acc: {results['CLM']['Accuracy']:.4f}")

    # ── TASK 2: Facts (T-REx) ─────────────────────────────────
    print(f"\n  [{label}] Facts Generation (T-REx)...")
    confs, accs = [], []

    for idx, row in trex_eval.iterrows():
        text   = trex_prompt + str(row['text']).strip()
        entity = str(row['entity']).strip()
        try:
            entity_toks = tokenizer(
                entity, add_special_tokens=False, return_tensors="pt"
            )["input_ids"]
            if entity_toks.shape[1] == 0: continue
            true_first = entity_toks[0, 0].item()

            context_ids = tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=512
            )["input_ids"].to(model.device)

            with torch.no_grad():
                probs = F.softmax(
                    model(input_ids=context_ids).logits[:, -1, :].float(), dim=-1
                )
            confs.append(probs.max(dim=-1).values.item())
            accs.append(int(probs.argmax(dim=-1).item() == true_first))
        except: continue

        if (idx+1) % 1000 == 0:
            print(f"    {idx+1}/{EVAL_SAMPLES} | "
                  f"ECE: {compute_ece(confs, accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f}")

    results['Facts'] = {
        'ECE':       compute_ece(confs, accs),
        'Accuracy':  float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ Facts ECE: {results['Facts']['ECE']:.4f} | "
          f"Acc: {results['Facts']['Accuracy']:.4f}")

    # ── TASK 3: MMLU ─────────────────────────────────────────
    print(f"\n  [{label}] MMLU...")
    confs, accs = [], []

    answer_token_ids = [
        tokenizer(opt, add_special_tokens=False,
                  return_tensors="pt")["input_ids"][0][0].item()
        for opt in ANSWER_OPTIONS
    ]

    def build_prompt(row):
        subject = row['subject']
        prompt  = (f"The following are multiple choice questions "
                   f"(with answers) about {subject}.\n\n")
        subject_val = mmlu_val_df[mmlu_val_df['subject'] == subject]
        if len(subject_val) < 5:
            subject_val = mmlu_val_df
        shots = subject_val.sample(n=min(5, len(subject_val)), random_state=SEED)
        for _, shot in shots.iterrows():
            ch = shot['choices']
            prompt += f"Question: {shot['question']}\n"
            for i, opt in enumerate(ANSWER_OPTIONS):
                prompt += f"{opt}. {ch[i]}\n"
            prompt += f"Answer: {ANSWER_OPTIONS[int(shot['answer'])]}\n\n"
        ch = row['choices']
        prompt += f"Question: {row['question']}\n"
        for i, opt in enumerate(ANSWER_OPTIONS):
            prompt += f"{opt}. {ch[i]}\n"
        prompt += "Answer:"
        return prompt

    for idx, row in mmlu_df.iterrows():
        try:
            input_ids = tokenizer(
                build_prompt(row), return_tensors="pt",
                truncation=True, max_length=2048
            )["input_ids"].to(model.device)

            with torch.no_grad():
                logits = model(input_ids=input_ids).logits[:, -1, :]

            ans_probs = F.softmax(
                torch.tensor([logits[0, tid].item()
                              for tid in answer_token_ids]).float(), dim=-1
            )
            confs.append(ans_probs.max().item())
            accs.append(int(ans_probs.argmax().item() == int(row['answer'])))
        except: continue

        if (idx+1) % 500 == 0:
            print(f"    {idx+1}/{len(mmlu_df)} | "
                  f"ECE: {compute_ece(confs, accs):.4f} | "
                  f"Acc: {np.mean(accs):.4f}")

    results['MMLU'] = {
        'ECE':       compute_ece(confs, accs),
        'Accuracy':  float(np.mean(accs)),
        'n_samples': len(confs),
        'bin_accs':  compute_reliability_diagram_data(confs, accs)[0],
        'bin_confs': compute_reliability_diagram_data(confs, accs)[1],
        'bin_sizes': compute_reliability_diagram_data(confs, accs)[2],
    }
    print(f"    ✅ MMLU ECE: {results['MMLU']['ECE']:.4f} | "
          f"Acc: {results['MMLU']['Accuracy']:.4f}")

    return results

print("Evaluation function ready.")

Evaluation function ready.


In [8]:
dolly_df = pd.read_csv(f'{DATA_PATH}/dolly/dolly_data.csv')
dolly_dataset = Dataset.from_pandas(dolly_df[['text']])

print(f"Dolly samples  : {len(dolly_dataset)}")
print(f"Sample:\n{dolly_df['text'].iloc[0][:300]}")

Dolly samples  : 15011
Sample:
### Instruction:
When did Virgin Australia start operating?
### Input:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, wit


In [9]:
print("Loading LLaMA-7B...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model + LoRA ready.")

Loading LLaMA-7B...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


trainable params: 4,194,304 || all params: 6,742,609,920 || trainable%: 0.0622
Model + LoRA ready.


In [10]:
training_args = SFTConfig(
    output_dir=f'{CKPT_PATH}/dolly_lora',
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,       # effective batch = 32
    learning_rate=3e-4,
    warmup_steps=100,
    lr_scheduler_type="linear",
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=3,
    report_to="none",
    dataloader_num_workers=2,
    dataset_text_field="text",
    max_length=2048,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dolly_dataset,
    args=training_args,
)

print("Starting Dolly LoRA training — Epoch 1...")
trainer.train()
print("Epoch 1 complete.")

Adding EOS to train dataset:   0%|          | 0/15011 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/15011 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting Dolly LoRA training — Epoch 1...


Step,Training Loss
50,1.550897
100,1.406722
150,1.370406
200,1.344140
250,1.371389
300,1.364909
350,1.329647
400,1.350470
450,1.329416


Epoch 1 complete.


In [11]:
save_path = f'{CKPT_PATH}/dolly_lora_epoch1'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Epoch 1 saved to: {save_path}")
print(f"Files: {os.listdir(save_path)}")

✅ Epoch 1 saved to: /kaggle/working/checkpoints/dolly_lora_epoch1
Files: ['tokenizer_config.json', 'adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json']
